In [17]:
import os
from dotenv import load_dotenv
from groq import Groq
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [5]:
load_dotenv()
groq_api_key = os.getenv('GROQ_API_KEY')

In [13]:
def get_completion(prompt, model="llama3-8b-8192"):
    client = Groq(api_key=groq_api_key)
    chat_completion = client.chat.completions.create(
                        messages = [{
                            "role":"user",
                            "content":prompt
                        }],
        model = model
    )
    return chat_completion.choices[0].message.content

In [8]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""
style = """American English \
in a calm and respectful tone
"""
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

In [16]:
print(get_completion(prompt, 'mixtral-8x7b-32768'))

I'm really upset that the lid of my blender flew off and splattered my kitchen walls with smoothie! To make matters worse, the warranty does not cover the cost of cleaning up my kitchen. I need your help right now, friend!


In [20]:
chat = ChatGroq(temperature=0, groq_api_key=groq_api_key, model_name="mixtral-8x7b-32768")
chat

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000025597AA5CD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025597AC0090>, temperature=1e-08, groq_api_key=SecretStr('**********'))

In [21]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [24]:
prompt_template = ChatPromptTemplate.from_template(template_string)
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [25]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [26]:
customer_style = """American English \
in a calm and respectful tone
"""
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [27]:
customer_messages = prompt_template.format_messages(
                    style = customer_style,
                    text = customer_email)

In [28]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [29]:
customer_messages[0]

HumanMessage(content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n")

In [30]:
response = chat(customer_messages)
print(response.content)

D:\anaconda\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


I'm really upset that the lid of my blender flew off and splattered my kitchen walls with smoothie! To make matters worse, the warranty does not cover the cost of cleaning up my kitchen. I need your help right now, friend!


In [37]:
system = "You are a helpful assistant."
human = "{text}"
prompt = ChatPromptTemplate.from_messages([("system", system), ("human", human)])

chain = prompt | chat
response = chain.invoke({"text": customer_messages})
print(response.content)

Sure, I'd be happy to help! The text you provided appears to be written in a form of English inspired by pirates. Here's a translation of the text in American English, spoken in a calm and respectful tone:

"I'm really upset that the lid of my blender came off and splattered my kitchen walls with smoothie! To make matters worse, the warranty doesn't cover the cost of cleaning up my kitchen. I need your help right now, friend!"
